In [0]:
%sql

DESCRIBE bakery_pos_develop.bronze.pos_order

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_pos_order AS
SELECT
    id AS pos_order_id,
    name AS pos_order_name,
    state AS pos_order_state,
    pos_reference,
    CAST(CASE
        WHEN account_move IS NULL OR account_move = 'false' THEN NULL
        ELSE get_json_object(account_move, '$[0]')
        END AS INT
    ) AS move_id,
    CAST(CASE
        WHEN account_move IS NULL OR account_move = 'false' THEN NULL
        ELSE get_json_object(account_move, '$[1]')
        END AS STRING
    ) AS move_name,
    session_id[0] AS session_id,
    user_id[0] AS user_id,
    CAST(CASE
        WHEN partner_id IS NULL OR partner_id = 'false' THEN NULL
        ELSE get_json_object(partner_id, '$[0]')
        END AS INT
    ) AS partner_id,
    CAST(date_order AS DATE) AS date_order,
    CAST(create_date AS TIMESTAMP) AS created_at,
    CAST(write_date AS TIMESTAMP) AS updated_at,
    processing_timestamp AS bronze_ingested_at
FROM bakery_pos_develop.bronze.pos_order
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY id
    ORDER BY write_date DESC,
    processing_timestamp DESC
) = 1


In [0]:
%sql

CREATE TABLE IF NOT EXISTS bakery_pos_develop.silver.pos_order (
    pos_order_id BIGINT,
    pos_order_name STRING,
    pos_order_state STRING,
    pos_reference STRING,
    move_id BIGINT,
    move_name STRING, 
    session_id BIGINT,
    user_id INT,
    partner_id BIGINT,
    date_order DATE,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    bronze_ingested_at TIMESTAMP,
    silver_processed_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN
) USING DELTA;

In [0]:
%sql

MERGE INTO bakery_pos_develop.silver.pos_order AS target
USING (
  SELECT *, current_timestamp() AS silver_processed_at 
  FROM stg_pos_order
) AS source
ON target.pos_order_id = source.pos_order_id
WHEN MATCHED AND source.updated_at > target.updated_at THEN
  UPDATE SET
    pos_order_name = source.pos_order_name,
    pos_order_state = source.pos_order_state,
    pos_reference = source.pos_reference,
    move_id = source.move_id,
    move_name = source.move_name,
    session_id = source.session_id,
    user_id = source.user_id,
    partner_id = source.partner_id,
    date_order = source.date_order,
    created_at = source.created_at,
    updated_at = source.updated_at,
    is_updated = True,
    is_deleted = False,
    bronze_ingested_at = source.bronze_ingested_at,
    silver_processed_at = source.silver_processed_at
WHEN NOT MATCHED THEN
  INSERT (
    pos_order_id,
    pos_order_name,
    pos_order_state,
    pos_reference,
    move_id,
    move_name,
    session_id,
    user_id,
    partner_id,
    date_order,
    created_at,
    updated_at,
    bronze_ingested_at,
    silver_processed_at,
    is_updated,
    is_deleted
  )
  VALUES (
    source.pos_order_id,
    source.pos_order_name,
    source.pos_order_state,
    source.pos_reference,
    source.move_id,
    source.move_name,
    source.session_id,
    source.user_id,
    source.partner_id,
    source.date_order,
    source.created_at,
    source.updated_at,
    source.bronze_ingested_at,
    source.silver_processed_at,
    False,
    False
  );